# SP04 — Multi-sample Spatial Integration

**Tools:** `moscot` (SpatialAlignmentProblem), `harmonypy`, `scanpy`  
**Dataset:** Mouse brain Visium H&E — two simulated slides  
**Papers:** [Lange et al. 2023, Nature Methods (Moscot)](https://doi.org/10.1038/s41592-023-02066-3) · [Zeira et al. 2022, Nature Methods (PASTE)](https://doi.org/10.1038/s41592-022-01459-6)

**Prerequisites:** SP00 (SpatialData) + SP01 (Visium pipeline). `../data/visium_processed.h5ad` required.

---

## Why multi-sample spatial is hard

Single-slide Visium analysis is straightforward. Multi-sample (multiple patients, time points, or serial sections) introduces challenges that don't exist in scRNA-seq:

1. **Physical misalignment** — each slide is placed differently on the array; coordinates are not comparable across slides
2. **Tissue shape variation** — even serial sections of the same tissue have slightly different shapes and orientations
3. **Batch effects** — same as scRNA-seq (sequencing run, library prep), but now confounded with spatial position
4. **Region correspondence** — "layer IV cortex" in patient 1 vs. patient 2 must be matched without cell barcodes

## Strategies

| Problem | Tool | Approach |
|---------|------|----------|
| Slide alignment (rigid) | `Moscot SpatialAlignmentProblem` | OT minimizing expression + spatial distance |
| Slide alignment (rigid) | PASTE | Procrustes alignment + OT |
| Batch correction (expression) | Harmony, scVI | Same as scRNA-seq |
| Cross-slide cell type mapping | Moscot, PASTE2 | OT between expression distributions |
| Multi-slide deconvolution | cell2location with batch | Jointly model multiple slides |

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import anndata as ad
import harmonypy as hm
import matplotlib.pyplot as plt

sc.settings.set_figure_params(dpi=80, facecolor='white')

## 1. Simulate Two Slides with Batch Effects

We create a second slide by applying a rotation, translation, and expression batch effect to the mouse brain Visium data — simulating a serial section or a replicate experiment.

In [ ]:
import os
if os.path.exists('../data/visium_processed.h5ad'):
    adata1 = sc.read_h5ad('../data/visium_processed.h5ad')
else:
    adata1 = sq.datasets.visium_hne_adata()
    sc.pp.filter_genes(adata1, min_cells=10)
    sc.pp.normalize_total(adata1, target_sum=1e4)
    sc.pp.log1p(adata1)
    sc.pp.pca(adata1)
    sc.pp.neighbors(adata1)
    sc.tl.leiden(adata1, resolution=0.4)
    sq.gr.spatial_neighbors(adata1, coord_type='visium')

adata1.obs['slide'] = 'slide1'
print('Slide 1:', adata1.shape)

In [ ]:
# Simulate slide 2: rotate 15°, translate, add batch noise
np.random.seed(42)
adata2 = adata1.copy()
adata2.obs['slide'] = 'slide2'

# --- Spatial transformation: rotate + translate ---
coords = adata1.obsm['spatial'].copy().astype(float)
center = coords.mean(axis=0)

# Rotation matrix (15 degrees)
angle = np.radians(15)
R = np.array([[np.cos(angle), -np.sin(angle)],
               [np.sin(angle),  np.cos(angle)]])
coords_rotated = (coords - center) @ R.T + center

# Translation: shift by (200, 150) pixels
coords_slide2 = coords_rotated + np.array([200, 150])
adata2.obsm['spatial'] = coords_slide2

# --- Expression batch effect: global scaling + per-gene noise ---
import scipy.sparse as sp
if sp.issparse(adata2.X):
    X2 = adata2.X.toarray()
else:
    X2 = adata2.X.copy()
X2 = X2 * 0.85 + np.random.normal(0, 0.05, X2.shape)  # scale down + add noise
adata2.X = sp.csr_matrix(X2)

# Re-run PCA on slide 2
sc.pp.pca(adata2)

print('Slide 2 (rotated + batch-shifted):', adata2.shape)

# Visualize both slides in original coordinates
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, adata, title in zip(axes, [adata1, adata2], ['Slide 1', 'Slide 2 (rotated+shifted)']):
    coords = adata.obsm['spatial']
    ax.scatter(coords[:, 0], coords[:, 1], c='steelblue', s=3, alpha=0.5)
    ax.set_title(title)
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 2. Joint Expression Analysis (Batch Correction)

The first step in any multi-slide analysis: pool the expression data and correct for slide-level batch effects. This is the same as multi-sample scRNA-seq (see `perturb_seq/14_batch_correction`).

In [ ]:
# Concatenate both slides
combined = ad.concat([adata1, adata2], keys=['slide1', 'slide2'], label='slide')
print('Combined:', combined.shape)

# Joint PCA on common HVGs
sc.pp.highly_variable_genes(combined, n_top_genes=2000, batch_key='slide')
sc.pp.pca(combined)
sc.pp.neighbors(combined)
sc.tl.umap(combined)

# Before correction: check batch mixing
sc.pl.umap(combined, color=['slide', 'leiden'],
           title=['Before correction: by slide', 'By cluster'])

In [ ]:
# Harmony batch correction
ho = hm.run_harmony(combined.obsm['X_pca'], combined.obs, vars_use='slide',
                    theta=1.5, random_state=42)
combined.obsm['X_harmony'] = ho.Z_corr.T

sc.pp.neighbors(combined, use_rep='X_harmony')
sc.tl.umap(combined)
sc.tl.leiden(combined, resolution=0.4, key_added='leiden_harmony')

sc.pl.umap(combined, color=['slide', 'leiden_harmony'],
           title=['After Harmony: by slide', 'Shared clusters'])

## 3. Spatial Alignment with Moscot

`theis_ecosystem/T07` introduced Moscot conceptually. Here we apply the `SpatialAlignmentProblem` to actually align the two slides' coordinate systems using optimal transport.

In [ ]:
try:
    from moscot.problems import SpatialAlignmentProblem
    HAS_MOSCOT = True
    print('Moscot available')
except ImportError:
    HAS_MOSCOT = False
    print('Install: pip install moscot')

In [ ]:
if HAS_MOSCOT:
    # Add a 'batch' column to distinguish slides in the combined object
    combined.obs['batch'] = combined.obs['slide'].map({'slide1': 0, 'slide2': 1}).astype(float)
    
    # SpatialAlignmentProblem: align two slides
    # alpha: trade-off between spatial distance cost and expression distance cost
    #   alpha=0: align purely on spatial coords (rigid registration)
    #   alpha=1: align purely on gene expression (expression matching)
    #   alpha=0.5: balanced (recommended default)
    sap = SpatialAlignmentProblem(adata=combined)
    sap = sap.prepare(
        batch_key='slide',
        spatial_key='spatial',
        joint_attr='X_pca',        # expression representation
    )
    sap = sap.solve(epsilon=0.01, alpha=0.5)
    
    # Apply alignment: transforms slide2 coordinates to match slide1
    combined = sap.align(reference_key='slide1')
    print('Alignment applied. Aligned coords in .obsm["spatial_aligned"]')
else:
    print('Moscot not installed. Using a manual Procrustes alignment as fallback.')

In [ ]:
if not HAS_MOSCOT:
    # Fallback: Procrustes alignment on shared spatial coordinates
    from scipy.spatial import procrustes
    
    coords1 = adata1.obsm['spatial'].astype(float)
    coords2 = adata2.obsm['spatial'].astype(float)
    
    # Subsample to equal size for Procrustes
    n = min(len(coords1), len(coords2))
    idx1 = np.random.choice(len(coords1), n, replace=False)
    idx2 = np.random.choice(len(coords2), n, replace=False)
    
    _, aligned2_sample, _ = procrustes(coords1[idx1], coords2[idx2])
    
    # Apply transformation to all slide2 spots (simple version)
    scale = np.std(aligned2_sample) / np.std(coords2[idx2])
    center2 = coords2.mean(0)
    center1 = coords1.mean(0)
    coords2_aligned = (coords2 - center2) * scale + center1
    
    # Store aligned coords
    slide2_mask = combined.obs['slide'] == 'slide2'
    combined.obsm['spatial_aligned'] = combined.obsm['spatial'].copy()
    combined.obsm['spatial_aligned'][slide2_mask] = coords2_aligned
    print('Procrustes alignment applied (fallback)')

In [ ]:
# Visualize before and after alignment
aligned_key = 'spatial_aligned' if 'spatial_aligned' in combined.obsm else 'spatial'

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Before
for slide, color in [('slide1', 'steelblue'), ('slide2', 'coral')]:
    mask = combined.obs['slide'] == slide
    c = combined[mask].obsm['spatial']
    axes[0].scatter(c[:, 0], c[:, 1], s=2, alpha=0.4, c=color, label=slide)
axes[0].set_title('Before alignment')
axes[0].legend(markerscale=5)
axes[0].set_aspect('equal')

# After
for slide, color in [('slide1', 'steelblue'), ('slide2', 'coral')]:
    mask = combined.obs['slide'] == slide
    c = combined[mask].obsm[aligned_key]
    axes[1].scatter(c[:, 0], c[:, 1], s=2, alpha=0.4, c=color, label=slide)
axes[1].set_title('After spatial alignment')
axes[1].legend(markerscale=5)
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

## 4. Cross-slide Spatial Differential Abundance

After alignment, we can test: in which spatial regions do cell type proportions differ between slides/conditions? This combines spatial alignment with Milo-style DA testing.

In [ ]:
# Test differential abundance per spatial region between slides
# First: what clusters are enriched in slide1 vs. slide2?
if 'leiden_harmony' not in combined.obs:
    sc.tl.leiden(combined, resolution=0.4, key_added='leiden_harmony')

# Simple test: chi-squared per cluster (or Fisher's exact for 2 slides)
from scipy.stats import chi2_contingency

results = []
for cluster in combined.obs['leiden_harmony'].unique():
    ct = pd.crosstab(
        combined.obs['leiden_harmony'] == cluster,
        combined.obs['slide'],
    )
    chi2, p, dof, expected = chi2_contingency(ct)
    
    # Observed vs expected enrichment in slide2
    n_slide1 = (combined.obs['slide'] == 'slide1').sum()
    n_slide2 = (combined.obs['slide'] == 'slide2').sum()
    n_cluster_slide1 = ((combined.obs['leiden_harmony'] == cluster) & (combined.obs['slide'] == 'slide1')).sum()
    n_cluster_slide2 = ((combined.obs['leiden_harmony'] == cluster) & (combined.obs['slide'] == 'slide2')).sum()
    
    log2fc = np.log2((n_cluster_slide2 / n_slide2 + 1e-6) / (n_cluster_slide1 / n_slide1 + 1e-6))
    results.append({'cluster': cluster, 'chi2': chi2, 'pvalue': p, 'log2FC': log2fc})

results_df = pd.DataFrame(results).sort_values('pvalue')
print('Spatial cluster enrichment (slide2 vs slide1):')
print(results_df.head(10))

In [ ]:
# Visualize the most differentially abundant cluster on each slide
top_cluster = results_df.iloc[0]['cluster']

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, slide in zip(axes, ['slide1', 'slide2']):
    mask = combined.obs['slide'] == slide
    ad_slide = combined[mask].copy()
    ad_slide.obsm['spatial'] = ad_slide.obsm[aligned_key]
    
    colors = ['#d62728' if c == top_cluster else 'lightgray'
              for c in ad_slide.obs['leiden_harmony']]
    coords = ad_slide.obsm['spatial']
    ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=3, alpha=0.6)
    ax.set_title(f'{slide}: cluster {top_cluster} highlighted')
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 5. Best Practices for Multi-sample Spatial Studies

In [ ]:
print("""
Recommended multi-sample spatial workflow:

1. Per-slide QC
   ─ Remove low-quality spots (pct_counts_mt, n_genes_by_counts)
   ─ Check spatial distribution of QC metrics per slide

2. Concatenate with slide metadata
   combined = ad.concat([s1, s2, s3], keys=['s1','s2','s3'], label='slide')

3. Joint HVG selection
   sc.pp.highly_variable_genes(combined, batch_key='slide')

4. Batch correction (expression)
   Option A: Harmony  → adata.obsm['X_harmony']
   Option B: scVI     → adata.obsm['X_scVI']  (better for many slides)

5. Shared clustering on corrected embedding
   sc.pp.neighbors(combined, use_rep='X_harmony')
   sc.tl.leiden(combined)

6. Spatial alignment of coordinates
   Option A: Moscot SpatialAlignmentProblem  (OT-based)
   Option B: Procrustes (simple rigid, no expression)
   Option C: PASTE / PASTE2 (OT + gene expression)

7. Cross-slide spatial analyses
   ─ Milo DA testing (perturb_seq/13_differential_abundance)
   ─ cell2location per slide then compare abundances
   ─ Region-matched DE (after anatomical annotation)

Key parameter:
   alpha in Moscot: 0=spatial only, 0.5=balanced, 1=expression only
   Start with alpha=0.5; increase if slides have different tissue coverage
""")

---
## Summary

| Challenge | Tool | Key parameter |
|-----------|------|---------------|
| Expression batch effects | `harmonypy`, `scvi-tools` | `batch_key='slide'` |
| Spatial coordinate alignment | `Moscot SpatialAlignmentProblem` | `alpha` (0=spatial, 1=expression) |
| Spatial alignment (rigid) | `scipy.spatial.procrustes` | — |
| Cross-slide DA testing | `milopy` (Milo) | `sample_col='slide'` |
| Multi-slide deconvolution | `cell2location` | add `batch_key` to both stages |

**Done with the Spatial Transcriptomics series!**

```
SP00 → SpatialData container (Zarr, coordinate systems)
SP01 → Visium in depth (spatial QC, regional DE, H&E features)
SP02 → Spatial domains (BANKSY, spatially-aware clustering)
SP03 → Sub-cellular resolution (seqFISH+, Xenium/MERFISH data model)
SP04 → Multi-sample integration (alignment, batch correction, cross-slide DA)
```

**Cross-series connections:**
- Cell type deconvolution per spot: `theis_ecosystem/T06` (cell2location)
- Cell-cell communication on spatial graph: `theis_ecosystem/T05` (LIANA)
- Optimal transport alignment theory: `theis_ecosystem/T07` (Moscot)